# Multi-Objective Optimization and Gait Analysis

Real walking design is a trade-off: a walker that goes far usually
wastes energy doing it. Single-objective optimization collapses that
trade-off into one number; **NSGA-II** keeps both on the table and
returns a Pareto front of non-dominated designs.

**What you'll learn:**
- Running `nsga_walking_optimization` with two physics objectives
- Reading a Pareto front and picking the best compromise
- `analyze_gait()`: duty factor, stride period, phase offsets
- Visualization helpers (`plot_pareto_front`, `plot_gait_diagram`)


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

import leggedsnake as ls

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.random.seed(42)


## 1. NSGA-II over distance and efficiency

Both objectives are physics-based (`DistanceFitness`,
`EfficiencyFitness`), so each candidate is a short pymunk rollout.
We keep `duration=2s`, `population_size=6`, `generations=3` to bound
the demo runtime — use order-of-magnitude larger settings for a
serious run.


In [ ]:
def walker_factory():
    return ls.Walker.from_jansen(scale=0.1)

prototype = walker_factory()
edge_names = list(prototype.dimensions.edge_distances.keys())
dim_values = list(prototype.dimensions.edge_distances.values())
lo = [0.7 * v for v in dim_values]
hi = [1.3 * v for v in dim_values]

cfg = ls.NsgaWalkingConfig(n_generations=3, pop_size=6, seed=42, verbose=False)

result = ls.nsga_walking_optimization(
    walker_factory=walker_factory,
    objectives=[
        ls.DistanceFitness(duration=2.0, n_legs=4),
        ls.EfficiencyFitness(duration=2.0, n_legs=4, min_distance=0.1),
    ],
    bounds=(lo, hi),
    objective_names=['distance', 'efficiency'],
    nsga_config=cfg,
    include_gait=False,
    include_stability=False,
)
print(f"Pareto front: {len(result.pareto_front.solutions)} solutions")
for i, sol in enumerate(result.pareto_front.solutions[:5]):
    print(f"  sol {i}: scores={[round(float(s), 3) for s in sol.scores]}")


## 2. Plot the Pareto front

`plot_pareto_front` is a thin matplotlib wrapper that highlights the
"best compromise" (closest to the utopia corner in objective space).


In [ ]:
fig = ls.plot_pareto_front(result, figsize=(6, 4))
plt.show()


## 3. Inspect the best-compromise solution

`result.best_compromise` is the Pareto solution with smallest L2
distance to the utopia corner in objective space.


In [ ]:
best = result.best_compromise()
print(f"best compromise scores: {best.scores}")
print(f"  dimensions: {dict(zip(edge_names, best.dimensions))}")


## 4. Gait analysis on the winner

`analyze_gait` takes the per-foot trajectories and returns **duty
factor** (fraction of the stride in stance), **stride period**, and
inter-foot **phase offsets**. Re-evaluating the winner with
`DistanceFitness(record_loci=True)` captures the trajectories we need.


In [ ]:
best_walker = walker_factory()
best_walker.set_num_constraints(list(best.dimensions))

fit_record = ls.DistanceFitness(duration=3.0, n_legs=4, record_loci=True)
fr = fit_record(best_walker.topology, best_walker.dimensions, ls.WorldConfig())
print(f"recorded distance={fr.score:.3f}, feet tracked={list(fr.loci.keys())}")

if fr.loci:
    foot_ids = list(fr.loci.keys())
    gait = ls.analyze_gait(fr.loci, foot_ids, dt=0.02)
    print(f"mean duty factor : {gait.mean_duty_factor:.2f}")
    print(f"mean stride freq : {gait.mean_stride_frequency:.2f} Hz")
    print(f"mean stride len  : {gait.mean_stride_length:.3f} m")


## 5. Gait diagram

`plot_gait_diagram` shows each foot as a stance/swing bar across
time. Symmetric gaits (trot, pace) produce simple alternating
patterns; chaotic gaits fragment.


In [ ]:
if fr.loci:
    fig = ls.plot_gait_diagram(gait, figsize=(8, 2.8))
    plt.show()
else:
    print("No recorded loci — skipping gait diagram")


## Summary

| Step                | API                                               |
|---------------------|---------------------------------------------------|
| Multi-obj search    | `nsga_walking_optimization(factory, objectives, bounds)` |
| Pareto plot         | `plot_pareto_front(result)`                       |
| Best compromise     | `result.best_compromise`                          |
| Record loci         | `DistanceFitness(record_loci=True)`               |
| Gait analysis       | `analyze_gait(loci, foot_ids)`                    |
| Stability           | `compute_stability_snapshot` / `StabilityTimeSeries` |

See `discover_walker.ipynb` for the next step — **topology
co-optimization**, where the search treats the mechanism's graph
structure itself as a decision variable (four-bar vs six-bar vs
eight-bar, determined by fitness rather than assumed).
